# Multilingual RAG Pipeline Summary With Memory Management

## Architecture Overview
- **Multilingual RAG system** supporting English, Urdu, and Arabic
- **Vector + Fulltext hybrid search** for retrieval
- **Semantic memory** for conversation context
- **Neo4j graph database** as vector store
- **Groq LLM** (Llama 3.1-8B) for generation

## Core Components

### 1. Data Ingestion
- PDF extraction using `PyPDF2`
- Text normalization for Arabic/Urdu scripts (diacritic removal, character mapping)
- Chunking with 800-character window size
- Embedding generation using `intfloat/multilingual-e5-large` (1024-dim vectors)
- Storage in Neo4j with `Chunk` nodes containing text and embeddings

### 2. Indexing Strategy
- **Vector index** named `pdf` on `Chunk.embedding` property
- **Fulltext index** named `ftPdfChunk` on `Chunk.text` property
- Unique constraints on chunk index

### 3. Retrieval Pipeline
- **Query expansion**: Generates original + Arabic + Urdu variants
- **Hybrid search** (70% vector + 30% fulltext weights)
- Score normalization using max score scaling
- Deduplication across expanded queries
- Returns top-5 most relevant chunks

### 4. Memory System
- **Semantic memory** storing Q&A pairs as embeddings
- Cosine similarity for memory retrieval
- Stores last 5 conversations (implicit limit)
- Used for query rewriting and context continuity

### 5. Query Processing
- **Query rewriting** with conversation memory context
- Language normalization (Arabic/Urdu script detection)
- Text cleaning (lowercase, diacritic removal for Arabic/Urdu)

### 6. Generation
- Context window: 3000 chars from chunks + 800 chars from memory
- Prompt engineering with language preservation rules
- Strict context-only answering (no outside knowledge)
- Temperature 0.3 for balanced creativity/accuracy
### Data Flow
- PDF → Extract → Normalize → Chunk → Embed → Neo4j
- ↓
- User Query → Expand → Rewrite → Hybrid Search → Retrieve
- ↓
- Memory → Context Building → LLM Generation → Answer

In [1]:
# ==========================================================
# MULTILINGUAL RAG WITH ADVANCED MEMORY (EN + UR + AR)
# ==========================================================

import re
import os
import numpy as np
from dotenv import load_dotenv

from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEmbeddings
from PyPDF2 import PdfReader
from groq import Groq

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(r".env", override=True)

hf_token = os.getenv("GROQ_API_KEY")

In [ ]:
# ==========================================================
# CONFIG
# ==========================================================

NEO4J_URI = ""
NEO4J_USER = ""
NEO4J_PASSWORD = ""


client = Groq(api_key=hf_token)

#Multilingual embedding model
hf_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

d:\MYFolder( Extra Effort)\LangChain Implementation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ==========================================================
# NORMALIZATION
# ==========================================================

def normalize_ar(text):
    text = re.sub("[ًٌٍَُِّْـ]", "", text)
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    return text

def normalize_ur(text):
    text = text.replace("ك", "ک").replace("ی", "ی")
    return text

def normalize_text(text):
    text = text.strip()
    if re.search(r'[\u0600-\u06FF]', text):
        text = normalize_ar(text)
        text = normalize_ur(text)
    return text.lower()

In [5]:
# ==========================================================
# PDF INGESTION
# ==========================================================

def read_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        text += normalize_text(page.extract_text() or "") + "\n"

    return text

In [6]:
# ==========================================================
# CHUNKING
# ==========================================================

def chunk_text(text, chunk_size=800):
    paragraphs = text.split("\n\n")
    chunks = []
    current = ""

    for p in paragraphs:
        if len(current) + len(p) < chunk_size:
            current += p + "\n"
        else:
            chunks.append(current.strip())
            current = p + "\n"

    if current:
        chunks.append(current.strip())

    return chunks

In [7]:
# ==========================================================
# EMBEDDING
# ==========================================================

def embed(texts):
    return hf_embeddings.embed_documents(texts)

In [8]:
# ==========================================================
# INDEX CREATION
# ==========================================================

def create_vector_index():
    driver.execute_query("""
    CREATE VECTOR INDEX pdf IF NOT EXISTS
    FOR (c:Chunk)
    ON c.embedding
    """)

def create_fulltext_index():
    driver.execute_query("""
    CREATE FULLTEXT INDEX ftPdfChunk IF NOT EXISTS
    FOR (c:Chunk)
    ON EACH [c.text]
    """)

In [9]:
# ==========================================================
# STORE DATA
# ==========================================================

def store_chunks(chunks, embeddings):

    query = """
    WITH $chunks as chunks, range(0, size($chunks)-1) AS index
    UNWIND index AS i
    WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding

    MERGE (c:Chunk {index: i})
    SET c.text = chunk,
        c.embedding = embedding
    """

    driver.execute_query(query, chunks=chunks, embeddings=embeddings)

In [10]:
# ==========================================================
# QUERY EXPANSION
# ==========================================================

def expand_query(query):
    return [
        query,
        f"{query} in Arabic",
        f"{query} in Urdu"
    ]

In [12]:
# ==========================================================
# HYBRID SEARCH
# ==========================================================

def hybrid_search(question, embedding, k=5):

    query = """
    CALL {

        CALL db.index.vector.queryNodes('chunk_index', $k, $embedding)
        YIELD node, score
        WITH collect({node:node, score:score}) AS nodes, max(score) AS max
        UNWIND nodes AS n
        RETURN n.node AS node, (n.score / max) * 0.7 AS score

        UNION

        CALL db.index.fulltext.queryNodes('chunk_ft', $question, {limit:$k})
        YIELD node, score
        WITH collect({node:node, score:score}) AS nodes, max(score) AS max
        UNWIND nodes AS n
        RETURN n.node AS node, (n.score / max) * 0.3 AS score
    }

    WITH node, max(score) AS score
    ORDER BY score DESC
    LIMIT $k

    RETURN node, score
    """

    records, _, _ = driver.execute_query(
        query,
        embedding=embedding,
        question=question,
        k=k
    )

    return records

In [13]:
# ==========================================================
# RETRIEVAL PIPELINE
# ==========================================================

def retrieve(question):

    expanded = expand_query(question)

    all_results = []

    for q in expanded:
        q = normalize_text(q)
        emb = embed([q])[0]
        results = hybrid_search(q, emb)
        all_results.extend(results)

    # deduplicate
    unique = {}
    for r in all_results:
        idx = r["node"]["index"]
        if idx not in unique or unique[idx]["score"] < r["score"]:
            unique[idx] = r

    return sorted(unique.values(), key=lambda x: x["score"], reverse=True)[:5]

In [14]:
# ==========================================================
# ADVANCED MEMORY (SEMANTIC)
# ==========================================================

conversation_history = []
memory_embeddings = []

def store_memory(question, answer):
    text = f"Q: {question} A: {answer}"
    emb = embed([text])[0]

    conversation_history.append(text)
    memory_embeddings.append(emb)


def retrieve_memory(query, top_k=1):

    if not memory_embeddings:
        return ""

    q_emb = embed([query])[0]

    scores = []
    for i, emb in enumerate(memory_embeddings):
        score = np.dot(q_emb, emb)
        scores.append((conversation_history[i], score))

    top = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

    return "\n".join([t[0] for t in top])

In [15]:
# ==========================================================
# QUERY REWRITING
# ==========================================================

def rewrite_query(question):

    memory_context = retrieve_memory(question)

    if not memory_context:
        return question

    prompt = f"""
Rewrite the question into a complete standalone query.

Conversation:
{memory_context}

Follow-up:
{question}

Rewritten:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [17]:
# ==========================================================
# GENERATION
# ==========================================================

# def generate_answer(question, records):

#     context = "\n".join([doc["node"]["text"] for doc in records])
#     memory_context = retrieve_memory(question)

#     prompt = f"""
# You are a multilingual assistant.

# Conversation:
# {memory_context}

# Context:
# {context}

# Question:
# {question}

# Rules:
# - Answer in same language
# - Use ONLY context
# - If missing say "I don't know"

# Answer:
# """

#     response = client.chat.completions.create(
#         model="llama-3.1-8b-instant",
#         messages=[{"role": "user", "content": prompt}],
#         temperature=0.3,
#         max_tokens=300
#     )

#     return response.choices[0].message.content
def generate_answer(question, records):

    def trim_text(text, max_chars):
        return text[:max_chars]

    #limit context
    context_parts = []
    for doc in records:
        context_parts.append(doc["node"]["text"][:1200])

    context = "\n\n".join(context_parts)
    context = trim_text(context, 3000)

    #limit memory
    memory_context = retrieve_memory(question)
    memory_context = trim_text(memory_context, 800)

    prompt = f"""
You are a multilingual assistant.

Conversation:
{memory_context}

Context:
{context}

Question:
{question}

Rules:
- Answer in same language
- Use ONLY context
- If missing say "I don't know"

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300
    )

    return response.choices[0].message.content

In [19]:
# ==========================================================
# MAIN LOOP
# ==========================================================

def main():

    # ====== RUN ONCE ======
    # text = read_pdf("CH1_SuraAlFatiha_2Edition.pdf")
    # chunks = chunk_text(text)
    # embeddings = embed(chunks)
    # create_vector_index()
    # create_fulltext_index()
    # store_chunks(chunks, embeddings)

    print("\nMultilingual RAG with Memory Ready!")
    while True:

        question = input("\nAsk (EN/UR/AR) or 'exit': ")
        if question.lower() == "exit":
            break

        #rewrite query
        standalone = rewrite_query(question)
        standalone = normalize_text(standalone)

        #retrieve
        records = retrieve(standalone)

        #generate
        answer = generate_answer(question, records)

        print("\n=== ANSWER ===\n")
        print(answer)

        #store memory
        store_memory(question, answer)


# ==========================================================
# RUN
# ==========================================================

if __name__ == "__main__":
    main()


Multilingual RAG with Memory Ready!


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL {\n\n        CALL db.index.vector.queryNodes('chunk_index', $k, $embedding)\n        YIELD node, score\n        WITH collect({node:node, score:score}) AS nodes, max(score) AS max\n        UNWIND nodes AS n\n        RETURN n.node AS node, (n.score / max) * 0.7 AS score\n\n        UNION\n\n        CALL db.index.fulltext.queryNodes('chun


=== ANSWER ===

Alhamdu lillah. 

Rehman is Allah (SWT). 

In the Tafseer Hub-e-Ali-ASWS, Al-Fatiha, it is mentioned that Allah is Rehman.


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL {\n\n        CALL db.index.vector.queryNodes('chunk_index', $k, $embedding)\n        YIELD node, score\n        WITH collect({node:node, score:score}) AS nodes, max(score) AS max\n        UNWIND nodes AS n\n        RETURN n.node AS node, (n.score / max) * 0.7 AS score\n\n        UNION\n\n        CALL db.index.fulltext.queryNodes('chun


=== ANSWER ===

Verse 1 hai: 

Alhamdu lillahi rabbi l'alamin.


...


=== ANSWER ===

Verse 1 hai:

Alhamdu lillahi rabbi l'alamin.

Verse 2 hai:

Ar rahmani rahim.

(Translation from Arabic to English: 
"The Beneficent, the Merciful.")

Verse 3 hai:

Maliki yawmi d-din.

(Translation from Arabic to English: 
"Lord of the Day of Judgment.")

Verse 4 hai:

Iyyaka na'budu wa iyyaka nasta'in.

(Translation from Arabic to English: 
"It is You we worship, and You we ask for help.")

Verse 5 hai:

Hudan wa nashiru al-huda wa na'lu al-zaalikeen.

(Translation from Arabic to English: 
"Guide us to the straight path. The path of those upon whom You have bestowed favor, not of those who have earned [Your] anger or of those who have strayed.")

Verse 6 hai:

Iyyaka na'budu wa iyyaka nasta'in.

(Translation from Arabic to English: 
"It is You we worship, and You we ask for help.")

Verse 7 hai:

Ihdina al-sirata al-mustaqim.

(Translation from Arabic to English: 
"Guide us to the straight path.")

Verse 8 hai:

Sirata alladhi na'mal lahu an'amta 'alayna.

(Translat

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL {\n\n        CALL db.index.vector.queryNodes('chunk_index', $k, $embedding)\n        YIELD node, score\n        WITH collect({node:node, score:score}) AS nodes, max(score) AS max\n        UNWIND nodes AS n\n        RETURN n.node AS node, (n.score / max) * 0.7 AS score\n\n        UNION\n\n        CALL db.index.fulltext.queryNodes('chun


=== ANSWER ===

الحمد لله رب العالمين.


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL {\n\n        CALL db.index.vector.queryNodes('chunk_index', $k, $embedding)\n        YIELD node, score\n        WITH collect({node:node, score:score}) AS nodes, max(score) AS max\n        UNWIND nodes AS n\n        RETURN n.node AS node, (n.score / max) * 0.7 AS score\n\n        UNION\n\n        CALL db.index.fulltext.queryNodes('chun


=== ANSWER ===

Verse 1 hai: 

Alhamdu lillahi rabbi l'alamin.
